<table style="border:0; background:none; width:100%">
<tr>
<td style="vertical-align: middle; width: 60%;">
<b>Pontificia Universidad Javeriana</b><br/>
Facultad de Ingeniería · Departamento de Ingeniería de Sistemas<br/>
<b>Procesamiento de Datos a Gran Escala</b>
</td>
<td style="vertical-align: middle; text-align: right; width: 40%;">
<b>Proyecto de Investigación — Entrega 2</b><br/>
Brecha digital y resultados Saber 11 en Colombia<br/>
<b>Grupo: REST pAPIs</b>
</td>
</tr>
</table>

---

# 09 — Aprendizaje no supervisado: K-Means de tipologías municipales

**Objetivo:** Agrupar municipios en tipologías según su perfil combinado de
**conectividad + pobreza + cobertura + rendimiento**, usando la vista minable
`gold/panel_municipal`.

**Algoritmo:** `KMeans` de Spark MLlib (sección 3 del notebook de clase).

**Elección de k:** método del codo (WCSS = `m.summary.trainingCost`) +
coeficiente de silueta (`ClusteringEvaluator`) para k ∈ [2, 8].

**Preprocesamiento idéntico al supervisado** (notebook 08): los modelos
no-supervisados también consumen la misma vista minable unificada.

## 1. Setup y datos

In [1]:
import sys, os
sys.path.insert(0, "/home/estudiante/proyecto_datos/scripts")
from common_spark import get_spark, P
from pyspark.sql import functions as F
from pyspark.sql.functions import col

from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

spark = get_spark("Entrega2-ML-KMeans", executor_memory="3g", driver_memory="3g", cores=2)
panel = spark.read.parquet(P.GOLD_PANEL_MUNICIPAL)

# Snapshot del año más reciente con tamaño mínimo
ano_max = panel.agg(F.max("ANO")).first()[0]
print(f"Año seleccionado para clustering: {ano_max}")
snap = (panel.filter(col("ANO") == ano_max)
             .filter(col("avg_punt_global").isNotNull())
             .filter(col("n_estudiantes") >= 30))
print(f"Filas (municipios) snapshot: {snap.count():,}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/22 08:23:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Año seleccionado para clustering: 2022


Filas (municipios) snapshot: 1,098


## 2. Selección de features y reporte de nulos

In [2]:
FEATURES = [
    "pct_internet_icfes",
    "accesos_per_capita_5_16",
    "idx_privacion",
    "pct_grupo_A",
    "avg_punt_global",
    "COBERTURA_NETA",
]
for f in FEATURES:
    snap = snap.withColumn(f, col(f).cast("double"))

null_counts = snap.select(
    [F.sum(col(c).isNull().cast("int")).alias(c) for c in FEATURES]
).first().asDict()
total = snap.count()
USABLE = [f for f in FEATURES if null_counts[f] < total]
print(f"Features usables (no 100% null): {len(USABLE)} / {len(FEATURES)}")
for f in FEATURES:
    pct = 100 * null_counts[f] / total
    flag = "DESCARTAR" if f not in USABLE else "ok"
    print(f"  {f:<28s} nulls={null_counts[f]:>5d}  {pct:>5.1f}%   {flag}")

Features usables (no 100% null): 6 / 6
  pct_internet_icfes           nulls=    0    0.0%   ok
  accesos_per_capita_5_16      nulls=    4    0.4%   ok
  idx_privacion                nulls=   12    1.1%   ok
  pct_grupo_A                  nulls=   12    1.1%   ok
  avg_punt_global              nulls=    0    0.0%   ok
  COBERTURA_NETA               nulls=    2    0.2%   ok


## 3. Imputación con mediana (approxQuantile + fillna — patrón clase 1.3)

In [3]:
medianas = {}
for f in USABLE:
    q = snap.approxQuantile(f, [0.5], 0.01)
    if q:
        medianas[f] = q[0]
print("Medianas imputadas:")
for f, m in medianas.items():
    print(f"  {f:<28s} = {m:.4f}")

snap_imp = snap.fillna(medianas)
nulos_post = snap_imp.select(
    [F.sum(col(c).isNull().cast("int")).alias(c) for c in USABLE]
).first().asDict()
print(f"\nNulos tras imputación: {sum(nulos_post.values())}")

Medianas imputadas:
  pct_internet_icfes           = 0.5000
  accesos_per_capita_5_16      = 0.6878
  idx_privacion                = 3.7701
  pct_grupo_A                  = 0.3773
  avg_punt_global              = 240.3400
  COBERTURA_NETA               = 0.8566



Nulos tras imputación: 0


## 4. Pipeline de feature engineering: Assembler + StandardScaler (clase 1.5)

In [4]:
assembler = VectorAssembler(inputCols=USABLE, outputCol="features_raw", handleInvalid="keep")
scaler    = StandardScaler(inputCol="features_raw", outputCol="features",
                            withMean=True, withStd=True)

prep_pipeline = Pipeline(stages=[assembler, scaler])
prep_model = prep_pipeline.fit(snap_imp)
data = (prep_model.transform(snap_imp)
        .select("COD_MPIO","COLE_DEPTO_UBICACION","features", *USABLE, "n_estudiantes")
        .cache())
print(f"Datos preparados: {data.count():,} municipios con vector 'features' listo.")

Datos preparados: 1,098 municipios con vector 'features' listo.


## 5. Elección de k — método del codo + silueta (clase 3.2)

In [5]:
results = []
ev = ClusteringEvaluator(featuresCol="features", predictionCol="cluster",
                          metricName="silhouette",
                          distanceMeasure="squaredEuclidean")

for k in range(2, 9):
    km = KMeans(featuresCol="features", predictionCol="cluster", k=k, seed=42, maxIter=40)
    m  = km.fit(data)
    df_pred = m.transform(data)
    wcss = m.summary.trainingCost              # WCSS
    silh = ev.evaluate(df_pred)                # silueta
    results.append((k, wcss, silh))
    print(f"  K={k}  →  WCSS={wcss:>10.2f}  |  Silhouette={silh:.4f}")

26/05/22 08:24:36 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


  K=2  →  WCSS=   4399.40  |  Silhouette=0.4493


  K=3  →  WCSS=   3573.68  |  Silhouette=0.4113


  K=4  →  WCSS=   3180.22  |  Silhouette=0.3183


  K=5  →  WCSS=   2917.36  |  Silhouette=0.2979


  K=6  →  WCSS=   2784.15  |  Silhouette=0.2866


  K=7  →  WCSS=   2579.93  |  Silhouette=0.2721


  K=8  →  WCSS=   2449.81  |  Silhouette=0.2650


In [6]:
# Plot codo + silueta (solo evidencia visual; cómputo ya está en Spark)
import matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
OUT_DIR = "/home/estudiante/proyecto_datos/evidencia"; os.makedirs(OUT_DIR, exist_ok=True)

ks = [r[0] for r in results]
wcss = [r[1] for r in results]
sils = [r[2] for r in results]

fig, axes = plt.subplots(1,2, figsize=(12,4))
axes[0].plot(ks, wcss, "o-", color="tab:blue")
axes[0].set_xlabel("k"); axes[0].set_ylabel("WCSS"); axes[0].set_title("Método del codo (WCSS)")
axes[0].grid(alpha=0.3)
axes[1].plot(ks, sils, "s-", color="tab:orange")
axes[1].set_xlabel("k"); axes[1].set_ylabel("Silhouette"); axes[1].set_title("Coeficiente de silueta")
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/kmeans_eleccion_k.png", dpi=110); plt.show()

## 6. Modelo final (k según mejor silueta)

In [7]:
K_BEST = max(results, key=lambda r: r[2])[0]
print(f"K elegido por silueta: {K_BEST}")

km_final = KMeans(featuresCol="features", predictionCol="cluster",
                   k=K_BEST, seed=42, maxIter=60)
model_km = km_final.fit(data)
clusters = model_km.transform(data).cache()
print(f"Tamaño por cluster:")
clusters.groupBy("cluster").count().orderBy("cluster").show()

K elegido por silueta: 2


Tamaño por cluster:


+-------+-----+
|cluster|count|
+-------+-----+
|      0|  603|
|      1|  495|
+-------+-----+



## 7. Perfilado de clusters (clase 3.3)

In [8]:
# Promedio por cluster para cada feature original (no escalada)
agg_exprs = [F.count("*").alias("n_municipios")]
if "avg_punt_global"           in USABLE: agg_exprs.append(F.round(F.avg("avg_punt_global"),1).alias("avg_punt"))
if "pct_internet_icfes"        in USABLE: agg_exprs.append(F.round(F.avg("pct_internet_icfes")*100,1).alias("pct_internet"))
if "accesos_per_capita_5_16"   in USABLE: agg_exprs.append(F.round(F.avg("accesos_per_capita_5_16"),3).alias("accesos_pc"))
if "idx_privacion"             in USABLE: agg_exprs.append(F.round(F.avg("idx_privacion"),2).alias("idx_priv"))
if "pct_grupo_A"               in USABLE: agg_exprs.append(F.round(F.avg("pct_grupo_A")*100,1).alias("pct_grupoA"))
if "COBERTURA_NETA"            in USABLE: agg_exprs.append(F.round(F.avg("COBERTURA_NETA")*100,1).alias("cob_neta"))

perfil = clusters.groupBy("cluster").agg(*agg_exprs).orderBy("cluster")
perfil.show()

# Centroides en el espacio escalado (mismo formato que la clase)
print("Centroides en el espacio de features escaladas:")
for i, c in enumerate(model_km.clusterCenters()):
    print(f"  Cluster {i}: {[round(x,3) for x in c]}")

+-------+------------+--------+------------+----------+--------+----------+--------+
|cluster|n_municipios|avg_punt|pct_internet|accesos_pc|idx_priv|pct_grupoA|cob_neta|
+-------+------------+--------+------------+----------+--------+----------+--------+
|      0|         603|   227.6|        38.6|     0.528|    4.39|      48.8|    82.8|
|      1|         495|   251.4|        64.2|     2.303|    3.23|      25.8|    90.3|
+-------+------------+--------+------------+----------+--------+----------+--------+

Centroides en el espacio de features escaladas:
  Cluster 0: [np.float64(-0.634), np.float64(-0.453), np.float64(0.589), np.float64(0.61), np.float64(-0.502), np.float64(-0.232)]
  Cluster 1: [np.float64(0.772), np.float64(0.551), np.float64(-0.718), np.float64(-0.743), np.float64(0.611), np.float64(0.283)]


## 8. Ejemplos de municipios por cluster

In [9]:
for c in sorted([r["cluster"] for r in clusters.select("cluster").distinct().collect()]):
    print(f"\n=== Cluster {c} (top 8 municipios por n_estudiantes) ===")
    (clusters.filter(col("cluster")==c)
             .select("COD_MPIO","COLE_DEPTO_UBICACION",
                     "avg_punt_global","pct_internet_icfes",
                     "idx_privacion","n_estudiantes")
             .orderBy(F.desc("n_estudiantes"))
             .show(8, truncate=False))


=== Cluster 0 (top 8 municipios por n_estudiantes) ===


+--------+--------------------+---------------+------------------+------------------+-------------+
|COD_MPIO|COLE_DEPTO_UBICACION|avg_punt_global|pct_internet_icfes|idx_privacion     |n_estudiantes|
+--------+--------------------+---------------+------------------+------------------+-------------+
|76109   |VALLE               |219.42         |0.5289            |4.278992157517104 |7385         |
|44001   |LA GUAJIRA          |233.56         |0.5723            |5.094320173178023 |5438         |
|44430   |LA GUAJIRA          |227.1          |0.3794            |4.467142648987324 |4563         |
|52835   |NARINO              |201.05         |0.334             |5.030325814536341 |4252         |
|05837   |ANTIOQUIA           |214.45         |0.5293            |4.281569494335451 |3650         |
|27001   |CHOCO               |220.62         |0.5256            |4.921386567626121 |3373         |
|13430   |BOLIVAR             |230.07         |0.5049            |4.829673423423423 |3294         |


## 9. Scatter de tipologías (pct_internet vs avg_punt, color por cluster)

In [10]:
plot_df = (clusters.select("pct_internet_icfes","avg_punt_global","cluster","n_estudiantes")
                   .toPandas().dropna())
fig, ax = plt.subplots(figsize=(9,6))
cmap = plt.get_cmap("tab10")
for c in sorted(plot_df["cluster"].unique()):
    sub = plot_df[plot_df["cluster"]==c]
    ax.scatter(sub["pct_internet_icfes"], sub["avg_punt_global"],
               s=sub["n_estudiantes"].clip(upper=2000)/8,
               alpha=0.55, color=cmap(c), label=f"Cluster {c}", edgecolors="none")
ax.set_xlabel("% estudiantes con internet (ICFES)")
ax.set_ylabel("Puntaje global promedio")
ax.set_title(f"K-Means k={K_BEST} — tipologías municipales (snapshot año {ano_max})")
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/kmeans_clusters_scatter.png", dpi=110); plt.show()

## 10. Persistir clusters y modelo en HDFS

In [11]:
CLUSTER_OUT = f"/usr/proyecto/gold/clusters_municipales_k{K_BEST}_ano{ano_max}"
(clusters.select("COD_MPIO","COLE_DEPTO_UBICACION","cluster", *USABLE, "n_estudiantes")
         .coalesce(1)
         .write.mode("overwrite").option("compression","snappy")
         .parquet(f"hdfs://10.43.97.164:9000{CLUSTER_OUT}"))

MODEL_PATH = f"/usr/proyecto/models/kmeans_k{K_BEST}"
model_km.write().overwrite().save(f"hdfs://10.43.97.164:9000{MODEL_PATH}")
print("Clusters:", CLUSTER_OUT)
print("Modelo  :", MODEL_PATH)

Clusters: /usr/proyecto/gold/clusters_municipales_k2_ano2022
Modelo  : /usr/proyecto/models/kmeans_k2


## 11. Conclusión

K-Means identifica tipologías de municipios sobre la vista minable Gold,
distinguiendo perfiles combinados (conectividad + pobreza + rendimiento). El
método del codo y la silueta convergen en el k seleccionado. Los centroides
escalados y el perfil promedio por cluster permiten interpretación de negocio
directa: cada cluster define un *grupo objetivo* del plan de acción del
Ministerio de Educación.

In [12]:
spark.stop()

26/05/22 08:25:07 WARN NioEventLoop: Selector.select() returned prematurely 512 times in a row; rebuilding Selector io.netty.channel.nio.SelectedSelectionKeySetSelector@20e68855.
